In [1]:
import os, time, subprocess, pythoncom, psutil
import polars as pl
import pandas as pd
from datetime import datetime, timedelta
from IPython.display import HTML, display

# ══════════════════════════════════════════════════════════════════════════════
# PATHS & LOAD
# ══════════════════════════════════════════════════════════════════════════════
first_glob = os.path.expanduser("~").replace("\\", "/")
ATD_PATH   = f"{first_glob}/Concentrix Corporation/WFM-Expedia-HCM - Branding files/BI_Task/CODE/Resources/ATD_Final.parquet"
PBI_URL    = "https://app.powerbi.com/view?r=eyJrIjoiOTVmMDY5MDItZTc1YS00ZjcyLThlOTctNTZmNmI4NTM4YzgwIiwidCI6IjU5OWU1MWQ2LTJmOGMtNDM0Ny04ZTU5LTFmNzk1YTUxYTk4YyIsImMiOjZ9&pageName=d135"

print("📂 Loading ATD_Final.parquet...")
atd_raw = pl.read_parquet(ATD_PATH)
print(f"✓ {atd_raw.shape[0]:,} rows")

# ══════════════════════════════════════════════════════════════════════════════
# CONFIG
# ══════════════════════════════════════════════════════════════════════════════
DISPLAY_NOTEBOOK = True
SEND_EMAIL       = True

EMAIL_TO = (
    "pradeep.bahadursha@concentrix.com;"
    "puneet.suneja@concentrix.com;"
    "kirpan.patar@concentrix.com"
)

EMAIL_CC = (
    "rahul.issar@concentrix.com;"
    "francesca.cioccari@concentrix.com;"
    "Varun.Kathuria@concentrix.com;"
    "urmila.chakka1@concentrix.com;"
    "ML.HOC.Expedia.Hierarchy@concentrix.com;"
    "EG_CAI_RAYAH_expedia_global_rtm@concentrix.com;"
    "VN_HOC_QUANG_vn_hcm_one_exp_wfm@concentrix.com;"
    "aas.mohammad@concentrix.com"
)

TARGET_PLANNED   = 4.0
TARGET_UNPLANNED = 6.0
TARGET_SHRINKAGE = 10.0

OVERRIDE_DATE = None #"2026-04-15"

# ─────────────────────────────────────────────────────────────────────────────
if OVERRIDE_DATE:
    report_date = datetime.strptime(OVERRIDE_DATE, "%Y-%m-%d")
    print(f"⚠️  OVERRIDE MODE: {OVERRIDE_DATE}")
else:
    report_date = datetime.now() - timedelta(days=1)

report_date_s = report_date.strftime("%d-%b-%Y")
report_date_d = report_date.date()

month_start   = report_date.replace(day=1).date()
current_month = report_date.strftime("%b-%y")

EMAIL_SUBJECT = f"Expedia VN - VN ATD Report Daily - {report_date_s}"

# ══════════════════════════════════════════════════════════════════════════════
# LOB MAPPING
# ══════════════════════════════════════════════════════════════════════════════
LOB_MAP   = {
    "Support_LG_Nesting":  "Lodging",
    "Lodging_Nesting":     "Lodging",
    "Support_NL_Nesting":  "Non_Lodging",
    "Non_Lodging_Nesting": "Non_Lodging",
}
KEEP_LOBS = ["Lodging", "Non_Lodging"]
LOB_ORDER = {"Lodging": 0, "Non_Lodging": 1}

atd = (
    atd_raw
    .with_columns(pl.col("LOB").replace(LOB_MAP).alias("LOB"))
    .filter(pl.col("LOB").is_in(KEEP_LOBS))
)

current_month = report_date.strftime("%b-%y")
d1_week_rows = atd.filter(pl.col("Date") == report_date_d)["Week Begin"].unique().to_list()
current_week = d1_week_rows[0] if d1_week_rows else atd.filter(
    pl.col("Date") <= report_date_d
)["Week Begin"].max()
current_week  = d1_week_rows[0] if d1_week_rows else atd["Week Begin"].max()
month_start   = report_date.replace(day=1).date()

atd_mtd = atd.filter(
    (pl.col("Month") == current_month) &
    (pl.col("Date") >= month_start) &
    (pl.col("Date") <= report_date_d)
)

atd_d1_week = atd.filter(pl.col("Week Begin") == current_week)

print(f"✓ D-1: {report_date_d} | Month: {current_month} | Week: {current_week}")
print(f"✓ MTD: {atd_mtd.shape[0]:,} | D-1 week: {atd_d1_week.shape[0]:,}")

# ══════════════════════════════════════════════════════════════════════════════
# COMPUTE HELPERS
# ══════════════════════════════════════════════════════════════════════════════
def _add_pct(df: pl.DataFrame) -> pl.DataFrame:
    hs = pl.col("HC Schedule"); pl_ = pl.col("HC Planned"); ul_ = pl.col("HC Unplanned")
    return df.with_columns([
        pl.when(hs>0).then((pl_/hs*100).round(1)).otherwise(None).alias("Planned (%)"),
        pl.when(hs>0).then((ul_/hs*100).round(1)).otherwise(None).alias("Unplanned (%)"),
        pl.when(hs>0).then(((pl_+ul_)/hs*100).round(2)).otherwise(None).alias("Shrinkage (%)"),
        pl.when(hs>0).then(((hs-pl_-ul_)/hs*100).round(2)).otherwise(None).alias("Attendance (%)"),
    ])

def _base_agg(frame, group_cols, present_alias="Present"):
    return frame.group_by(group_cols).agg([
        pl.col("HC Schedule").cast(pl.Float64,strict=False).fill_null(0).sum().alias("HC Schedule"),
        pl.col("Present").cast(pl.Float64,strict=False).fill_null(0).sum().alias(present_alias),
        pl.col("Planned").cast(pl.Float64,strict=False).fill_null(0).sum().alias("HC Planned"),
        pl.col("Unplanned").cast(pl.Float64,strict=False).fill_null(0).sum().alias("HC Unplanned"),
    ])

def _make_subtotal(df, lob, label_col):
    sub = df[df["LOB"]==lob]
    hs  = sub["HC Schedule"].sum()
    row = {"LOB": lob, label_col: f"{lob} — Subtotal", "_row_type": "subtotal"}
    for c in ["HC Schedule","Present","HC Present","HC Present (ATD)","HC Planned","HC Unplanned"]:
        if c in sub.columns: row[c] = sub[c].sum()
    if hs > 0:
        pl_ = row.get("HC Planned",0); ul_ = row.get("HC Unplanned",0)
        row["Planned (%)"]    = round(pl_/hs*100, 1)
        row["Unplanned (%)"]  = round(ul_/hs*100, 1)
        row["Shrinkage (%)"]  = round((pl_+ul_)/hs*100, 2)
        row["Attendance (%)"] = round((hs-pl_-ul_)/hs*100, 2)
    return row

def _make_grand(df, label_col, extra=None):
    hs  = df["HC Schedule"].sum()
    row = {label_col: "Grand Total", "_row_type": "total"}
    if extra: row.update(extra)
    for c in ["HC Schedule","Present","HC Present","HC Present (ATD)","HC Planned","HC Unplanned"]:
        if c in df.columns: row[c] = df[c].sum()
    if hs > 0:
        pl_ = row.get("HC Planned",0); ul_ = row.get("HC Unplanned",0)
        row["Planned (%)"]    = round(pl_/hs*100, 1)
        row["Unplanned (%)"]  = round(ul_/hs*100, 1)
        row["Shrinkage (%)"]  = round((pl_+ul_)/hs*100, 2)
        row["Attendance (%)"] = round((hs-pl_-ul_)/hs*100, 2)
    return row

def _inject_lob(df, label_col, sort_within=None):
    result = []
    for lob in ["Lodging","Non_Lodging"]:
        grp = df[df["LOB"]==lob].copy()
        if sort_within: grp = grp.sort_values(sort_within)
        grp["_row_type"] = "data"
        result.append(grp)
        result.append(pd.DataFrame([_make_subtotal(df, lob, label_col)]))
    result.append(pd.DataFrame([_make_grand(df, label_col)]))
    out = pd.concat(result, ignore_index=True)
    out["_row_type"] = out["_row_type"].fillna("data")
    return out

def compute_day_wise(frame):
    agg = _add_pct(
        _base_agg(frame, ["Month","LOB","Date"], "HC Present (ATD)")
        .sort([pl.col("LOB").replace(LOB_ORDER).cast(pl.Int32), pl.col("Date")])
    ).to_pandas()
    agg["Date"] = pd.to_datetime(agg["Date"]).dt.strftime("%Y-%m-%d")
    agg["_row_type"] = "data"
    return _inject_lob(agg, "Date", sort_within=["Date"])

def compute_day_wise_total(frame):
    agg = _add_pct(
        _base_agg(frame, ["Month", "Date"], "HC Present (ATD)")
        .sort(pl.col("Date"))
    ).to_pandas()
    agg["Date"] = pd.to_datetime(agg["Date"]).dt.strftime("%Y-%m-%d")
    agg["_row_type"] = "data"

    # Grand Total row
    grand = _make_grand(agg, "Date", {"Month": agg["Month"].iloc[0] if len(agg) else ""})
    return pd.concat([agg, pd.DataFrame([grand])], ignore_index=True)

def compute_sup_wise(frame):
    agg = _add_pct(
        _base_agg(frame, ["Month","Supervisor Name"])
        .sort("Supervisor Name")
    ).to_pandas()
    agg["_row_type"] = "data"
    grand = _make_grand(agg, "Supervisor Name", {"Month": agg["Month"].iloc[0] if len(agg) else ""})
    return pd.concat([agg, pd.DataFrame([grand])], ignore_index=True)

def compute_shift_wise(frame):
    valid = frame.filter(
        pl.col("Original.Shift").str.contains("-") |
        pl.col("Original.Shift").is_in(["AL","CO","LWP"])
    )
    agg = _add_pct(
        _base_agg(valid, ["Month","LOB","Original.Shift"])
        .sort([pl.col("LOB").replace(LOB_ORDER).cast(pl.Int32), pl.col("Original.Shift")])
    ).to_pandas()
    agg["_row_type"] = "data"
    return _inject_lob(agg, "Original.Shift", sort_within=["Original.Shift"])

def compute_tl_wise(frame):
    agg = _add_pct(
        _base_agg(frame, ["LOB","Week Begin","Supervisor Name"], "HC Present")
        .sort([pl.col("LOB").replace(LOB_ORDER).cast(pl.Int32), pl.col("Supervisor Name")])
    ).to_pandas()
    agg["_row_type"] = "data"
    return _inject_lob(agg, "Supervisor Name", sort_within=["Supervisor Name"])

def compute_shift_wise_d1(frame):
    valid = frame.filter(
        pl.col("Original.Shift").str.contains("-") |
        pl.col("Original.Shift").is_in(["AL","CO","LWP"])
    )
    agg = _add_pct(
        _base_agg(valid, ["LOB","Original.Shift"], "HC Present")
        .sort([pl.col("LOB").replace(LOB_ORDER).cast(pl.Int32), pl.col("Original.Shift")])
    ).to_pandas()
    agg["_row_type"] = "data"
    return _inject_lob(agg, "Original.Shift", sort_within=["Original.Shift"])

def compute_absence(frame):
    return (
        frame.filter(
            (pl.col("Date") == report_date_d) &
            ~pl.col("Attendance").is_in(["PR","WO","WFH"]) &
            pl.col("Attendance").is_not_null() &
            (pl.col("Original.Shift") != "Termination")
        )
        .select([
            pl.col("OracleID"), pl.col("Employee Name"),
            pl.col("Supervisor Name"), pl.col("LOB"),
            pl.col("Original.Shift").alias("First Shift"),
            pl.col("Wave"),
            pl.col("Start Time").alias("Start"),
            pl.col("End Time").alias("End"),
            pl.col("Duration").cast(pl.Float64, strict=False),
            pl.col("HC Schedule").cast(pl.Float64, strict=False),
            pl.col("Present").cast(pl.Float64, strict=False).alias("HC Present"),
            pl.col("SUM Productive").cast(pl.Float64, strict=False).alias("Sum Productive"),
            pl.col("Attendance"),
            pl.col("Reason").alias("Reasons"),
        ])
        .sort([pl.col("LOB").replace(LOB_ORDER).cast(pl.Int32),
               pl.col("Supervisor Name"), pl.col("Employee Name")])
        .to_pandas()
    )

# ══════════════════════════════════════════════════════════════════════════════
# STYLE CONSTANTS
# ══════════════════════════════════════════════════════════════════════════════
HDR_DARK = "#1a3a5c"
HDR_MID  = "#1f5c99"
HDR_LITE = "#2e86c1"
YLW_HDR  = "#FFD700"
BLU_ROW  = "#dce8f5"
WHT_ROW  = "#ffffff"
TOT_BG   = "#1a3a5c"
SUB_BG   = "#2e5f8a"
MET_BG   = "#d4f4e2"; MET_FG  = "#1a5c2a"
MISS_BG  = "#fde8ea"; MISS_FG = "#9b1c2a"
FONT     = "font-family:Arial,sans-serif;font-size:11px;"

# th: center aligned header
TH_S  = f"{FONT}padding:5px 8px;color:#fff;font-weight:bold;white-space:nowrap;text-align:center;border:1px solid rgba(255,255,255,0.2);"
# td: left aligned, fit content
TD_S  = f"{FONT}padding:4px 8px;border:1px solid #dce8f5;white-space:nowrap;text-align:left;"
TD_TOT= f"{FONT}padding:4px 8px;white-space:nowrap;text-align:left;background:{TOT_BG};color:#fff;font-weight:bold;border:1px solid rgba(255,255,255,0.12);"
TD_SUB= f"{FONT}padding:4px 8px;white-space:nowrap;text-align:left;background:{SUB_BG};color:#fff;font-weight:bold;font-style:italic;border:1px solid rgba(255,255,255,0.15);"

CSS = f"""
body{{margin:0;padding:16px;background:#fff;font-family:Arial,sans-serif}}
.section{{margin-bottom:24px}}
.t{{border-collapse:collapse;font-size:11px;font-family:Arial,sans-serif;
   white-space:nowrap;width:auto}}
.t thead th{{padding:5px 8px;color:#fff;font-weight:bold;
   text-align:center;border:1px solid rgba(255,255,255,0.2);white-space:nowrap}}
.t tbody td{{padding:4px 8px;border:1px solid #dce8f5;
   text-align:left;white-space:nowrap}}
.t tbody tr.blu td{{background:{BLU_ROW}}}
.t tbody tr.wht td{{background:{WHT_ROW}}}
.t tbody tr.sub td{{background:{SUB_BG}!important;color:#fff!important;
   font-weight:bold!important;font-style:italic;
   border:1px solid rgba(255,255,255,0.15)!important}}
.t tbody tr.tot td{{background:{TOT_BG}!important;color:#fff!important;
   font-weight:bold!important;border:1px solid rgba(255,255,255,0.12)!important}}
.met{{background:{MET_BG}!important;color:{MET_FG}!important;font-weight:bold!important}}
.miss{{background:{MISS_BG}!important;color:{MISS_FG}!important;font-weight:bold!important}}
.legend{{font-size:10.5px;margin:4px 0 10px;color:#555}}
.legend span{{padding:2px 8px;margin-right:6px;font-weight:bold;border-radius:3px}}
"""

# ══════════════════════════════════════════════════════════════════════════════
# HTML MICRO HELPERS
# ══════════════════════════════════════════════════════════════════════════════
def _th(lbl, bg=HDR_MID):
    return f'<th style="{TH_S}background:{bg};">{lbl}</th>'

def _th_l(lbl, bg=HDR_DARK):
    return f'<th style="{TH_S}background:{bg};">{lbl}</th>'

def fv(v, pct=False):
    if v is None or (isinstance(v, float) and pd.isna(v)): return "&#8212;"
    if pct:  return f"{v:.1f}%"
    if isinstance(v, float): return f"{v:,.1f}"
    return str(v)

def _pct_cls(v, target):
    if v is None or (isinstance(v, float) and pd.isna(v)): return ""
    return "miss" if v > target else "met"

def _inline_c(cls):
    if cls == "met":  return f"background:{MET_BG};color:{MET_FG};font-weight:bold;"
    if cls == "miss": return f"background:{MISS_BG};color:{MISS_FG};font-weight:bold;"
    return ""

def _legend(for_email=False):
    s  = f"{FONT}font-size:10.5px;margin:4px 0 10px;"
    sp = "padding:2px 8px;margin-right:6px;font-weight:bold;border-radius:3px;"
    txt = (
        f'<span style="{sp}background:{MET_BG};color:{MET_FG}">&#9632; Met target</span>'
        f'<span style="{sp}background:{MISS_BG};color:{MISS_FG}">&#9632; Missed target</span>'
        f'&nbsp;|&nbsp;Targets: Planned &le;{TARGET_PLANNED:.0f}% &nbsp;'
        f'Unplanned &le;{TARGET_UNPLANNED:.0f}% &nbsp;'
        f'Shrinkage &le;{TARGET_SHRINKAGE:.0f}%'
    )
    if for_email:
        return f'<p style="{s}color:#555;">{txt}</p>'
    return f'<div class="legend">{txt}</div>'

# ══════════════════════════════════════════════════════════════════════════════
# DATA BAR
# ══════════════════════════════════════════════════════════════════════════════
def _data_bar(val, col_max, color: str, for_email=False):
    if val is None or (isinstance(val, float) and pd.isna(val)) or col_max == 0:
        return "", None
    ratio = max(0.0, min(1.0, val / col_max))
    if ratio == 0:
        return "", None
    num_str = f"{val:,.1f}" if isinstance(val, float) else str(val)
    pct = ratio * 100

    if not for_email:
        style = (
            f"background:linear-gradient(to right,{color} {pct:.1f}%,"
            f"#ffffff {pct:.1f}%);color:#000;font-weight:bold;"
        )
        return style, None
    else:
        total_w = 80
        bar_w   = max(2, int(ratio * total_w))
        rest_w  = total_w - bar_w
        txt_c = "#fff"
        html = (
            f'<table border="0" cellspacing="0" cellpadding="0" '
            f'style="border-collapse:collapse;width:{total_w}px;display:inline-table;">'
            f'<tr>'
            f'<td width="{bar_w}" style="background:{color};height:16px;'
            f'vertical-align:middle;padding:0 3px;white-space:nowrap;">'
            f'<span style="font-family:Arial,sans-serif;font-size:10px;'
            f'color:{txt_c};font-weight:bold;">{num_str}</span>'
            f'</td>'
            f'<td width="{rest_w}" style="background:#f5f5f5;height:16px;'
            f'font-size:1px;">&nbsp;</td>'
            f'</tr></table>'
        )
        return "", html

def _build_heat_maps(df: pd.DataFrame) -> dict:
    data = df[df["_row_type"]=="data"] if "_row_type" in df.columns else df
    result = {}
    for col, color in [("HC Planned","#f39c12"),("HC Unplanned","#e74c3c")]:
        if col in data.columns:
            vals = pd.to_numeric(data[col], errors="coerce").dropna()
            result[col] = {"max": vals.max() if len(vals) else 0, "color": color}
    return result

# ══════════════════════════════════════════════════════════════════════════════
# GENERIC TABLE RENDERER
# ══════════════════════════════════════════════════════════════════════════════
def _render(df: pd.DataFrame, cols: list, for_email=False) -> str:
    t_cls = "" if for_email else 'class="t" '
    heat  = _build_heat_maps(df)

    # table: width:auto + table-layout:auto = fit content
    tbl_style = (
        "border-collapse:collapse;width:auto;table-layout:auto;"
        f"{FONT}"
    )
    h = [f'<table {t_cls}style="{tbl_style}"><thead><tr>']

    for lbl,bg,is_l,_,__,___ in cols:
        h.append(_th_l(lbl,bg) if is_l else _th(lbl,bg))
    h.append('</tr></thead><tbody>')

    alt = 0
    for _, row in df.iterrows():
        rt = row.get("_row_type","data")
        if rt == "data":
            even   = alt % 2 == 0; alt += 1
            bg_r   = BLU_ROW if even else WHT_ROW
            tr_cls = "blu" if even else "wht"
        else:
            bg_r   = SUB_BG if rt == "subtotal" else TOT_BG
            tr_cls = "sub" if rt == "subtotal" else "tot"

        h.append('<tr>' if for_email else f'<tr class="{tr_cls}">')

        for lbl, bg, is_l, col_key, is_pct, target in cols:
            val = row.get(col_key, "")
            v   = fv(val, is_pct)

            # ── subtotal / total ─────────────────────────────────────────────
            if rt in ("subtotal","total"):
                td_s = TD_SUB if rt == "subtotal" else TD_TOT
                h.append(f'<td style="{td_s}">{v}</td>')
                continue

            # ── PCT color ────────────────────────────────────────────────────
            pct_cls = _pct_cls(val, target) if target is not None else ""
            pct_ic  = _inline_c(pct_cls)

            # ── Data bar ─────────────────────────────────────────────────────
            heat_style = ""
            heat_html  = None
            if col_key in heat and not is_pct:
                hm      = heat[col_key]
                num_val = pd.to_numeric(val, errors="coerce")
                if not pd.isna(num_val):
                    heat_style, heat_html = _data_bar(
                        num_val, hm["max"], hm["color"], for_email=for_email
                    )

            # ── Render td ────────────────────────────────────────────────────
            if for_email:
                if pct_ic:
                    h.append(f'<td style="{TD_S}background:{bg_r};{pct_ic}">{v}</td>')
                elif heat_html:
                    h.append(
                        f'<td style="{TD_S}background:{bg_r};padding:1px 4px;'
                        f'vertical-align:middle;">{heat_html}</td>'
                    )
                else:
                    h.append(f'<td style="{TD_S}background:{bg_r};">{v}</td>')
            else:
                if pct_cls:
                    h.append(f'<td class="{pct_cls}" style="{TD_S}">{v}</td>')
                elif heat_style:
                    h.append(f'<td style="{TD_S}{heat_style}">{v}</td>')
                else:
                    h.append(f'<td style="{TD_S}background:{bg_r};">{v}</td>')

        h.append('</tr>')
    h.append('</tbody></table>')
    return "".join(h)

# ══════════════════════════════════════════════════════════════════════════════
# ABSENCE TABLE
# ══════════════════════════════════════════════════════════════════════════════
def _build_absence_table(df: pd.DataFrame, for_email=False) -> str:
    ATD_COLORS = {
        "HAL": ("#FFFACD","#7a5200"),
        "UAL": (MISS_BG,  MISS_FG),
        "SL":  (BLU_ROW,  HDR_DARK),
        "CO":  ("#e8f5e9","#1a5c2a"),
        "AL":  (BLU_ROW,  HDR_DARK),
        "LWP": ("#f3e5f5","#6a1b9a"),
    }
    TXT_COLS = ["OracleID","Employee Name","Supervisor Name","LOB","First Shift","Wave"]
    CTR_COLS = ["Start","End"]
    NUM_COLS = ["Duration","HC Schedule","HC Present","Sum Productive"]

    t_cls     = "" if for_email else 'class="t" '
    tbl_style = f"border-collapse:collapse;width:auto;table-layout:auto;{FONT}"
    h = [f'<table {t_cls}style="{tbl_style}"><thead><tr>']
    for lbl,bg in zip(
        TXT_COLS + CTR_COLS + NUM_COLS + ["Attendance","Reasons"],
        [HDR_DARK]*len(TXT_COLS) + [HDR_MID]*(len(CTR_COLS)+len(NUM_COLS)) + [HDR_MID,HDR_LITE]
    ):
        h.append(f'<th style="{TH_S}background:{bg};">{lbl}</th>')
    h.append('</tr></thead><tbody>')

    for i, row in df.iterrows():
        even  = i % 2 == 0
        bg_r  = BLU_ROW if even else WHT_ROW
        tr_cls= "blu" if even else "wht"
        h.append('<tr>' if for_email else f'<tr class="{tr_cls}">')

        for col in TXT_COLS:
            v = str(row.get(col,"")) if row.get(col) is not None else "&#8212;"
            h.append(f'<td style="{TD_S}background:{bg_r};">{v}</td>')

        for col in CTR_COLS:
            val = row.get(col)
            v = str(val) if val is not None and not (isinstance(val,float) and pd.isna(val)) else "&#8212;"
            h.append(f'<td style="{TD_S}background:{bg_r};text-align:center;">{v}</td>')

        for col in NUM_COLS:
            v = fv(row.get(col))
            h.append(f'<td style="{TD_S}background:{bg_r};text-align:right;">{v}</td>')

        atd_val       = str(row.get("Attendance",""))
        bg_a, fg_a    = ATD_COLORS.get(atd_val, ("#fff","#000"))
        h.append(
            f'<td style="{TD_S}background:{bg_a};color:{fg_a};font-weight:bold;text-align:center;">'
            f'{atd_val}</td>'
        )
        reasons = str(row.get("Reasons","")) if row.get("Reasons") is not None else "&#8212;"
        h.append(f'<td style="{TD_S}background:{bg_r};">{reasons}</td>')
        h.append('</tr>')
    h.append('</tbody></table>')
    return "".join(h)

# ══════════════════════════════════════════════════════════════════════════════
# COLUMN SCHEMAS  (label, bg, is_left, col_key, is_pct, target)
# ══════════════════════════════════════════════════════════════════════════════
C_DAY = [
    ("Month",          HDR_DARK, True,  "Month",            False, None),
    ("LOB",            HDR_DARK, True,  "LOB",              False, None),
    ("Date",           HDR_DARK, True,  "Date",             False, None),
    ("HC Schedule",    HDR_MID,  False, "HC Schedule",      False, None),
    ("HC Present(ATD)",HDR_MID,  False, "HC Present (ATD)", False, None),
    ("HC Planned",     HDR_MID,  False, "HC Planned",       False, None),
    ("HC Unplanned",   HDR_MID,  False, "HC Unplanned",     False, None),
    ("Planned (%)",    HDR_LITE, False, "Planned (%)",      True,  TARGET_PLANNED),
    ("Unplanned (%)",  HDR_LITE, False, "Unplanned (%)",    True,  TARGET_UNPLANNED),
    ("Shrinkage (%)",  HDR_LITE, False, "Shrinkage (%)",    True,  TARGET_SHRINKAGE),
]
C_DAY_TOTAL = [
    ("Month",           HDR_DARK, True,  "Month",            False, None),
    ("Date",            HDR_DARK, True,  "Date",             False, None),
    ("HC Schedule",     HDR_MID,  False, "HC Schedule",      False, None),
    ("HC Present(ATD)", HDR_MID,  False, "HC Present (ATD)", False, None),
    ("HC Planned",      HDR_MID,  False, "HC Planned",       False, None),
    ("HC Unplanned",    HDR_MID,  False, "HC Unplanned",     False, None),
    ("Planned (%)",     HDR_LITE, False, "Planned (%)",      True,  TARGET_PLANNED),
    ("Unplanned (%)",   HDR_LITE, False, "Unplanned (%)",    True,  TARGET_UNPLANNED),
    ("Shrinkage (%)",   HDR_LITE, False, "Shrinkage (%)",    True,  TARGET_SHRINKAGE),
]
C_SUP = [
    ("Month",          HDR_DARK, True,  "Month",           False, None),
    ("Supervisor Name",HDR_DARK, True,  "Supervisor Name", False, None),
    ("HC Schedule",    HDR_MID,  False, "HC Schedule",     False, None),
    ("Present",        HDR_MID,  False, "Present",         False, None),
    ("HC Planned",     HDR_MID,  False, "HC Planned",      False, None),
    ("HC Unplanned",   HDR_MID,  False, "HC Unplanned",    False, None),
    ("Planned (%)",    HDR_LITE, False, "Planned (%)",     True,  TARGET_PLANNED),
    ("Unplanned (%)",  HDR_LITE, False, "Unplanned (%)",   True,  TARGET_UNPLANNED),
    ("Shrinkage (%)",  HDR_LITE, False, "Shrinkage (%)",   True,  TARGET_SHRINKAGE),
]
C_SHF = [
    ("Month",          HDR_DARK, True,  "Month",           False, None),
    ("LOB",            HDR_DARK, True,  "LOB",             False, None),
    ("Original Shift", HDR_DARK, True,  "Original.Shift",  False, None),
    ("HC Schedule",    HDR_MID,  False, "HC Schedule",     False, None),
    ("Present",        HDR_MID,  False, "Present",         False, None),
    ("HC Planned",     HDR_MID,  False, "HC Planned",      False, None),
    ("HC Unplanned",   HDR_MID,  False, "HC Unplanned",    False, None),
    ("Planned (%)",    HDR_LITE, False, "Planned (%)",     True,  TARGET_PLANNED),
    ("Unplanned (%)",  HDR_LITE, False, "Unplanned (%)",   True,  TARGET_UNPLANNED),
    ("Shrinkage (%)",  HDR_LITE, False, "Shrinkage (%)",   True,  TARGET_SHRINKAGE),
]
C_TL = [
    ("LOB",            HDR_DARK, True,  "LOB",             False, None),
    ("Week Begin",     HDR_DARK, True,  "Week Begin",      False, None),
    ("Supervisor Name",HDR_DARK, True,  "Supervisor Name", False, None),
    ("HC Schedule",    HDR_MID,  False, "HC Schedule",     False, None),
    ("HC Present",     HDR_MID,  False, "HC Present",      False, None),
    ("HC Planned",     HDR_MID,  False, "HC Planned",      False, None),
    ("HC Unplanned",   HDR_MID,  False, "HC Unplanned",    False, None),
    ("Planned (%)",    HDR_LITE, False, "Planned (%)",     True,  TARGET_PLANNED),
    ("Unplanned (%)",  HDR_LITE, False, "Unplanned (%)",   True,  TARGET_UNPLANNED),
    ("Shrinkage (%)",  HDR_LITE, False, "Shrinkage (%)",   True,  TARGET_SHRINKAGE),
    ("Attendance (%)", HDR_LITE, False, "Attendance (%)",  True,  None),
]
C_SHF_D1 = [
    ("LOB",            HDR_DARK, True,  "LOB",             False, None),
    ("Original Shift", HDR_DARK, True,  "Original.Shift",  False, None),
    ("HC Schedule",    HDR_MID,  False, "HC Schedule",     False, None),
    ("HC Present",     HDR_MID,  False, "HC Present",      False, None),
    ("HC Planned",     HDR_MID,  False, "HC Planned",      False, None),
    ("HC Unplanned",   HDR_MID,  False, "HC Unplanned",    False, None),
    ("Planned (%)",    HDR_LITE, False, "Planned (%)",     True,  TARGET_PLANNED),
    ("Unplanned (%)",  HDR_LITE, False, "Unplanned (%)",   True,  TARGET_UNPLANNED),
    ("Shrinkage (%)",  HDR_LITE, False, "Shrinkage (%)",   True,  TARGET_SHRINKAGE),
    ("Attendance (%)", HDR_LITE, False, "Attendance (%)",  True,  None),
]

# ══════════════════════════════════════════════════════════════════════════════
# SECTION / GROUP HELPERS
# ══════════════════════════════════════════════════════════════════════════════
def _grp_hdr(title, subtitle, color, for_email=False):
    title_s = f"{FONT}font-size:14px;font-weight:bold;color:#fff;margin:0;"
    sub_s   = f"{FONT}font-size:11px;color:#ffffff;margin:3px 0 0;"
    inner   = f'<p style="{title_s}">{title}</p><p style="{sub_s}">{subtitle}</p>'
    if for_email:
        return (
            f'<table width="100%" border="0" cellspacing="0" cellpadding="0" '
            f'style="margin:18px 0 8px;">'
            f'<tr><td style="background:{color};padding:10px 14px;border-radius:4px;">'
            f'{inner}</td></tr></table>'
        )
    return (
        f'<div style="background:{color};padding:10px 14px;border-radius:4px;'
        f'margin:18px 0 8px;">{inner}</div>'
    )

def _tbl_hdr(badge, note, for_email=False):
    badge_s = (f"background:{YLW_HDR};color:#000;font-weight:bold;font-size:12px;"
               f"padding:3px 10px;border-radius:3px;")
    note_s  = (f"{FONT}font-size:10.5px;color:#555;background:#f8f8f8;"
               f"border-left:3px solid {HDR_MID};padding:4px 10px;"
               f"margin:4px 0 6px;border-radius:0 3px 3px 0;")
    if for_email:
        return (f'<p style="margin:12px 0 2px;">'
                f'<span style="{badge_s}">{badge}</span></p>'
                f'<p style="{note_s}">{note}</p>')
    return (f'<div style="margin:12px 0 2px;">'
            f'<span style="{badge_s}">{badge}</span></div>'
            f'<div style="{note_s}">{note}</div>')

def _sec(badge, note, df, cols, for_email=False):
    return (
        _tbl_hdr(badge, note, for_email)
        + _render(df, cols, for_email)
        + _legend(for_email)
    )

# ══════════════════════════════════════════════════════════════════════════════
# BUILD ALL SECTIONS
# ══════════════════════════════════════════════════════════════════════════════
def build_all_sections(for_email=False):
    GRP_MTD = "#1f5c99"
    GRP_D1  = "#5c1f99"
    parts   = []

    # GROUP 1: MTD
    parts.append(_grp_hdr(
        f"📅 Month-to-Date Attendance — {current_month}",
        f"Period: {month_start.strftime('%d-%b-%Y')} → {report_date_s} &nbsp;|&nbsp; "
        f"Includes: Day Wise, Supervisor Wise, Shift Wise",
        GRP_MTD, for_email
    ))
    parts.append(_sec(
        "1. DAY WISE",
        f"Daily headcount by LOB from {month_start.strftime('%d-%b-%Y')} to {report_date_s}. "
        f"Subtotal per LOB, Grand Total at bottom.",
        df_day, C_DAY, for_email
    ))
    parts.append(_sec(
        "1. DAY WISE — TOTAL (All LOBs Combined)",
        f"Daily headcount summary from {month_start.strftime('%d-%b-%Y')} to {report_date_s}. "
        f"All LOBs combined — Grand Total at bottom.",
        df_day_total, C_DAY_TOTAL, for_email
    ))
    parts.append(_sec(
        "2. SUPERVISOR WISE",
        "MTD aggregation per Team Leader. Planned = approved leave, Unplanned = unexpected absence.",
        df_sup, C_SUP, for_email
    ))
    parts.append(_sec(
        "3. SHIFT WISE",
        "MTD breakdown by shift (shifts with '-', AL, CO, LWP). Subtotal per LOB.",
        df_shift, C_SHF, for_email
    ))

    # GROUP 2: Previous Date
    parts.append(_grp_hdr(
        f"📆 Previous Date Attendance — {report_date_s}",
        f"Week: {current_week} &nbsp;|&nbsp; "
        f"Includes: Teamleader Wise, Shift Wise, Detailed Absence",
        GRP_D1, for_email
    ))
    parts.append(_sec(
        f"4. TEAMLEADER WISE — {report_date_s}",
        f"Headcount by Supervisor for week {current_week}. "
        f"Attendance % = (HC Schedule - Planned - Unplanned) / HC Schedule.",
        df_tl, C_TL, for_email
    ))
    parts.append(_sec(
        f"5. SHIFT WISE — {report_date_s}",
        f"Shift breakdown for week {current_week}. Same shift filter as MTD Shift Wise.",
        df_sw_d1, C_SHF_D1, for_email
    ))
    parts.append(
        _tbl_hdr(
            f"6. DETAILED ABSENCE — {report_date_s}",
            "Agents marked absent (excluding PR/WO/WFH and Termination shifts). "
            "Includes Start/End shift, Duration, HC Schedule, HC Present, and Sum Productive. "
            "Sorted by LOB → Supervisor → Agent Name.",
            for_email
        )
        + _build_absence_table(df_abs, for_email)
    )
    return "".join(parts)

# ══════════════════════════════════════════════════════════════════════════════
# EMAIL GREETING / SIGNATURE
# ══════════════════════════════════════════════════════════════════════════════
def build_atd_greeting():
    return f"""
<p style="{FONT}font-size:12px;margin:0 0 10px;line-height:1.7">Dear Team,</p>
<p style="{FONT}font-size:12px;margin:0 0 10px;line-height:1.7">
    I would like to share the attendance report as of <strong>{report_date_s}</strong>.
</p>
<p style="{FONT}font-size:12px;margin:0 0 16px;line-height:1.7">
    BI Link: &nbsp;
    <a href="{PBI_URL}" target="_blank"
       style="display:inline-block;background:#F2C811;color:#000000;
              font-weight:bold;font-size:11px;text-decoration:none;
              padding:4px 10px;border-radius:3px;border:1px solid #c9a800;
              font-family:Arial,sans-serif;">
        &#128269; Power BI Dashboard
    </a>
</p>
<p style="{FONT}font-size:12px;margin:0 0 6px;font-weight:bold;">
    &#128203; Sheet: ATD &nbsp;&nbsp;
    <span style="color:{HDR_MID}">&#8594; MTD + Previous Date breakdown below</span>
</p>
<hr style="border:none;border-top:1px solid #e0e0e0;margin:10px 0 16px;">
"""

def build_atd_signature():
    return f"""
<hr style="border:none;border-top:1px solid #e0e0e0;margin:16px 0 10px;">
<p style="{FONT}font-size:12px;margin:0 0 4px;line-height:1.7">Thanks &amp; Regards,</p>
<p style="{FONT}font-size:12px;font-weight:bold;margin:0 0 2px;">Chinh Nguyen</p>
<p style="{FONT}font-size:11px;color:#555;margin:0 0 2px;">Analyst, WFM Real Time Management</p>
<br>
<p style="{FONT}font-size:11px;color:#555;margin:0 0 2px;line-height:1.6">
    Level 4, Tower 1, OneHub Saigon, Lot C1-2, D1 Street, Saigon Hi Tech Park,<br>
    Tan Phu Ward, District 9, Ho Chi Minh City, Vietnam
</p>
<p style="{FONT}font-size:11px;color:#555;margin:0 0 2px;">
    Ph No: +84 986 473 419 &nbsp;|&nbsp;
    Email: <a href="mailto:huuchinh.nguyen@concentrix.com"
       style="color:{HDR_MID};font-weight:bold;text-decoration:none;">
       huuchinh.nguyen@concentrix.com</a>
</p>
<p style="{FONT}font-size:10px;color:#aaa;margin-top:10px;">
    Generated: {datetime.now().strftime("%Y-%m-%d %H:%M")} &nbsp;|&nbsp;
    Source: ATD_Final.parquet
</p>
"""

# ══════════════════════════════════════════════════════════════════════════════
# COMPUTE ALL
# ══════════════════════════════════════════════════════════════════════════════
print("⏳ Computing tables...")
df_day   = compute_day_wise(atd_mtd)
df_day_total = compute_day_wise_total(atd_mtd)
df_sup   = compute_sup_wise(atd_mtd)
df_shift = compute_shift_wise(atd_mtd)
df_tl    = compute_tl_wise(atd_d1_week)
df_sw_d1 = compute_shift_wise_d1(atd_d1_week)
df_abs   = compute_absence(atd)
print("✓ All computed")

# ══════════════════════════════════════════════════════════════════════════════
# DISPLAY NOTEBOOK
# ══════════════════════════════════════════════════════════════════════════════
if DISPLAY_NOTEBOOK:
    nb_html = (
        "<!DOCTYPE html><html><head><meta charset='utf-8'>"
        f"<style>{CSS}</style></head><body>"
        + build_all_sections(for_email=False)
        + "</body></html>"
    )
    escaped = nb_html.replace("&","&amp;").replace('"',"&quot;").replace("'","&#39;")
    display(HTML(
        f'<iframe srcdoc="{escaped}" style="width:100%;border:none;min-height:900px;" '
        f'onload="this.style.height=(this.contentDocument.body.scrollHeight+40)+\'px\'"></iframe>'
    ))
    print("✓ Display done")

# ══════════════════════════════════════════════════════════════════════════════
# SEND EMAIL
# ══════════════════════════════════════════════════════════════════════════════
if SEND_EMAIL:
    import win32com.client

    email_html = (
        "<!--[if mso]><xml><o:OfficeDocumentSettings><o:AllowPNG/>"
        "<o:PixelsPerInch>96</o:PixelsPerInch></o:OfficeDocumentSettings></xml><![endif]-->"
        f"<div style='padding:20px 24px;background:#fff;{FONT}'>"
        + build_atd_greeting()
        + build_all_sections(for_email=True)
        + build_atd_signature()
        + "</div>"
    )

    def send_auto(to, cc, subject, html_body, quit_after=True):
        pythoncom.CoInitialize()
        was_on = any(p.name().lower()=="outlook.exe" for p in psutil.process_iter(["name"]))
        if not was_on:
            for exe in [
                r"C:\Program Files\Microsoft Office\root\Office16\OUTLOOK.EXE",
                r"C:\Program Files (x86)\Microsoft Office\root\Office16\OUTLOOK.EXE",
            ]:
                if os.path.exists(exe): subprocess.Popen([exe]); break
            print("⏳ Starting Outlook...")
            for _ in range(30):
                time.sleep(1)
                try: win32com.client.GetActiveObject("Outlook.Application"); break
                except: pass
        try:
            ol = win32com.client.Dispatch("Outlook.Application")
            ol.GetNamespace("MAPI").Logon()
            mail = ol.CreateItem(0)
            mail.To=to; mail.CC=cc; mail.Subject=subject; mail.HTMLBody=html_body
            mail.Send()
            print(f"✓ Sent → {to}")
            time.sleep(3)
        finally:
            if quit_after and not was_on:
                try: ol.Quit(); print("✓ Outlook closed")
                except: pass

    send_auto(EMAIL_TO, EMAIL_CC, EMAIL_SUBJECT, email_html, quit_after=True)

📂 Loading ATD_Final.parquet...
✓ 32,180 rows
✓ D-1: 2026-08-05 | Month: Aug-26 | Week: WB0308
✓ MTD: 740 | D-1 week: 1,036
⏳ Computing tables...
✓ All computed


c:\Users\huuchinh.nguyen\AppData\Local\anaconda3\Lib\site-packages\IPython\core\display.py:431: UserWarning: Consider using IPython.display.IFrame instead
  warnings.warn("Consider using IPython.display.IFrame instead")


Month,LOB,Date,HC Schedule,HC Present(ATD),HC Planned,HC Unplanned,Planned (%),Unplanned (%),Shrinkage (%)
Aug-26,Lodging,2026-08-01,72.0,63.0,4.0,5.0,5.6%,6.9%,12.5%
Aug-26,Lodging,2026-08-02,66.0,55.5,5.0,5.5,7.6%,8.3%,15.9%
Aug-26,Lodging,2026-08-03,49.0,42.5,3.0,3.5,6.1%,7.1%,13.3%
Aug-26,Lodging,2026-08-04,62.0,54.5,2.0,5.5,3.2%,8.9%,12.1%
Aug-26,Lodging,2026-08-05,69.0,62.0,3.0,4.0,4.3%,5.8%,10.1%
&#8212;,Lodging,Lodging — Subtotal,318.0,277.5,17.0,23.5,5.3%,7.4%,12.7%
Aug-26,Non_Lodging,2026-08-01,14.0,13.0,1.0,0.0,7.1%,0.0%,7.1%
Aug-26,Non_Lodging,2026-08-02,8.0,7.0,1.0,0.0,12.5%,0.0%,12.5%
Aug-26,Non_Lodging,2026-08-03,16.0,13.0,2.0,1.0,12.5%,6.3%,18.8%
Aug-26,Non_Lodging,2026-08-04,19.0,17.0,1.0,1.0,5.3%,5.3%,10.5%


✓ Display done
✓ Sent → pradeep.bahadursha@concentrix.com;puneet.suneja@concentrix.com;kirpan.patar@concentrix.com


In [2]:
# ══════════════════════════════════════════════════════════════════════════════
# PATHS & LOAD
# ══════════════════════════════════════════════════════════════════════════════
first_glob = os.path.expanduser("~").replace("\\", "/")
ATD_PATH   = f"{first_glob}/Concentrix Corporation/WFM-Expedia-HCM - Branding files/BI_Task/CODE/Resources/ATD_Final.parquet"

print("📂 Loading ATD_Final.parquet...")
atd_raw = pl.read_parquet(ATD_PATH)
print(f"✓ {atd_raw.shape[0]:,} rows")

# ══════════════════════════════════════════════════════════════════════════════
# CONFIG
# ══════════════════════════════════════════════════════════════════════════════
MISMATCH_LOOKBACK_DAYS = 15   # ← chỉnh số ngày lookback cho Historical Mismatch tại đây

EMAIL_TO = (
    "puneet.suneja@concentrix.com;"
    "kirpan.patar@concentrix.com;"
    "ML.HOC.Expedia.Hierarchy@concentrix.com"
)

EMAIL_CC = (
    "Varun.Kathuria@concentrix.com;"
    "urmila.chakka1@concentrix.com;"
    "VN_HOC_QUANG_vn_hcm_one_exp_wfm@concentrix.com;"
    "ExpediaVN_Training_Team@concentrix.com;"
    "ExpediaVN_QA_Team@concentrix.com"
)

# ─────────────────────────────────────────────────────────────────────────────
if OVERRIDE_DATE:
    report_date   = datetime.strptime(OVERRIDE_DATE, "%Y-%m-%d")
    print(f"⚠️  OVERRIDE MODE: using {OVERRIDE_DATE}")
else:
    report_date   = datetime.now() - timedelta(days=1)
    print(f"✓ AUTO MODE: using D-1 = {report_date.strftime('%Y-%m-%d')}")

report_date_s = report_date.strftime("%d-%b-%Y")
report_date_d = report_date.date()

def _ordinal(n):
    s = {1:"st",2:"nd",3:"rd"}.get(n%10 if n%100 not in (11,12,13) else 0,"th")
    return f"{n}{s}"

day_ord       = _ordinal(report_date.day)
month_yr      = report_date.strftime("%b'%y")
EMAIL_SUBJECT = f"Expedia VN - Attendance Mismatch & NM Report - as of the {day_ord} of {month_yr}"
print(f"✓ Subject: {EMAIL_SUBJECT}")

# ══════════════════════════════════════════════════════════════════════════════
# LOB MAPPING  — support* → Support, non* → Non_Lodging, lodging* → Lodging
# ══════════════════════════════════════════════════════════════════════════════
atd = atd_raw.with_columns(
    pl.when(pl.col("LOB").str.to_lowercase().str.contains("support"))
      .then(pl.lit("Support"))
      .when(pl.col("LOB").str.to_lowercase().str.contains("non"))
      .then(pl.lit("Non_Lodging"))
      .when(pl.col("LOB").str.to_lowercase().str.contains("lodging"))
      .then(pl.lit("Lodging"))
      .otherwise(pl.col("LOB"))
      .alias("LOB")
)

atd = atd.filter(
    ~pl.col("Original.Shift").is_in(["Termination", "Training"]) &
    ~pl.col("Original.Shift").str.to_lowercase().str.contains("flex")
)

LOB_ORDER = {"Lodging": 0, "Non_Lodging": 1, "Support": 2}

# ══════════════════════════════════════════════════════════════════════════════
# RAMCO MISMATCH LOGIC  (mirror DAX SWITCH)
# ══════════════════════════════════════════════════════════════════════════════
LEAVE_CODES = ["AB", "SL", "LWP", "AL"]

def compute_ramco_check(frame: pl.DataFrame) -> pl.DataFrame:
    att   = pl.col("Attendance")
    ramco = pl.col("Ramco Marked")
    shift = pl.col("Original.Shift")

    check = (
        # ── TRUE cases ────────────────────────────────────────────────────────
        # CO + WO → TRUE
        pl.when((att == "CO") & (ramco == "WO")).then(pl.lit("TRUE"))
        # WO/OFF + PO → TRUE
        .when(att.is_in(["WO", "OFF", "CO"]) & (ramco == "PO")).then(pl.lit("TRUE"))
        # FIX: Original.Shift contains HAL + Ramco = HAL → TRUE  ← thêm dòng này
        .when(shift.str.contains("HAL") & (ramco == "HAL")).then(pl.lit("TRUE"))

        # ── FALSE cases ───────────────────────────────────────────────────────
        # WO/OFF + PR → FALSE
        .when(att.is_in(["WO", "OFF"]) & (ramco == "PR")).then(pl.lit("FALSE"))
        # HAL ↔ PR mismatch → FALSE
        .when((att == "HAL") & (ramco == "PR")).then(pl.lit("FALSE"))
        .when((att == "PR")  & (ramco == "HAL")).then(pl.lit("FALSE"))
        # FIX: HAL + full-day leave code → FALSE
        .when((att == "HAL") & ramco.is_in(LEAVE_CODES)).then(pl.lit("FALSE"))
        # FIX: full-day leave code + HAL ramco → FALSE
        .when(att.is_in(LEAVE_CODES) & (ramco == "HAL")).then(pl.lit("FALSE"))
        # AB + WO → FALSE
        .when((att == "AB") & (ramco == "WO")).then(pl.lit("FALSE"))

        # ── NM (pending) ──────────────────────────────────────────────────────
        .when(
            (att != "NCNS") &
            (ramco.is_null() | (ramco == "") | (ramco == "NM"))
        ).then(pl.lit("NM"))

        # ── TRUE cases (continued) ────────────────────────────────────────────
        .when(att == ramco).then(pl.lit("TRUE"))
        .when(att.is_in(LEAVE_CODES) & ramco.is_in(LEAVE_CODES)).then(pl.lit("TRUE"))
        .when(
            (att == "NCNS") &
            (ramco.is_null() | (ramco == "") | (ramco == "NM"))
        ).then(pl.lit("TRUE"))
        .when((att == "PR") & ramco.is_in(["PO", "PH"])).then(pl.lit("TRUE"))
        .when((att.is_null() | (att == "")) & (ramco == "HO")).then(pl.lit("TRUE"))
        .when(
            ~shift.str.contains("-") &
            ramco.is_not_null() & (ramco != "") &
            ~att.is_in(["WO", "CO", "HO", "HAL", "OFF"])
        ).then(pl.lit("TRUE"))

        .otherwise(pl.lit("FALSE"))
    )
    return frame.with_columns(check.alias("Ramco_Check"))

# ══════════════════════════════════════════════════════════════════════════════
# COMPUTE MISMATCH (FALSE cases)
# ══════════════════════════════════════════════════════════════════════════════
MISMATCH_COLS = [
    "Date", "Supervisor Name", "IEX ID", "OracleID", "Employee Name", "LOB",
    "Original.Shift", "Attendance", "Ramco Marked", "Ramco_Check",
    "Start Time", "End Time", "Remark",
    "HC Schedule", "Present", "Planned", "Unplanned", "Duration", "SUM Productive",
]

def _to_mismatch_df(polars_result: pl.DataFrame) -> pd.DataFrame:
    """Shared post-processing: to_pandas + date format + rename columns."""
    df = polars_result.to_pandas()
    if "Date" in df.columns:
        df["Date"] = pd.to_datetime(df["Date"]).dt.strftime("%m/%d/%Y")
    return df.rename(columns={
        "Ramco_Check":    "Ramco_Mismatch",
        "Remark":         "Remark (Str)",
        "SUM Productive": "Prod Hour",
    })

def compute_mismatch(frame: pl.DataFrame) -> pd.DataFrame:
    """FALSE cases cho D-1."""
    checked = compute_ramco_check(frame)
    result  = (
        checked
        .filter(pl.col("Ramco_Check") == "FALSE")
        .filter(pl.col("LOB") != "Support")     
        .select([c for c in MISMATCH_COLS if c in checked.columns])
        .sort([
            pl.col("LOB").replace_strict(LOB_ORDER, default=99).cast(pl.Int32),
            pl.col("Supervisor Name"),
            pl.col("Employee Name"),
        ])
    )
    return _to_mismatch_df(result)

def compute_historical_mismatch(frame_full: pl.DataFrame, d1: datetime) -> pd.DataFrame:
    """FALSE cases từ D-{MISMATCH_LOOKBACK_DAYS} đến D-2 (không lấy D-1)."""
    d2_date        = (d1 - timedelta(days=1)).date()
    lookback_start = (d1 - timedelta(days=MISMATCH_LOOKBACK_DAYS)).date()

    historical = frame_full.filter(
        (pl.col("Date") >= lookback_start) &
        (pl.col("Date") <= d2_date)
    )
    if historical.is_empty():
        return pd.DataFrame()

    checked = compute_ramco_check(historical)
    result  = (
        checked
        .filter(pl.col("Ramco_Check") == "FALSE")
        .filter(pl.col("LOB") != "Support")
        .select([c for c in MISMATCH_COLS if c in checked.columns])
        .sort([
            pl.col("Date"),   # sort theo ngày trước để nhóm rõ ràng
            pl.col("LOB").replace_strict(LOB_ORDER, default=99).cast(pl.Int32),
            pl.col("Supervisor Name"),
            pl.col("Employee Name"),
        ])
    )
    if result.is_empty():
        return pd.DataFrame()
    return _to_mismatch_df(result)

# ══════════════════════════════════════════════════════════════════════════════
# COMPUTE NM PENDING (last 15 days pivot)
# ══════════════════════════════════════════════════════════════════════════════
def compute_nm_pending(frame_full: pl.DataFrame, d1: datetime) -> pd.DataFrame:
    lookback_start = (d1 - timedelta(days=15)).date()
    recent = frame_full.filter(
        (pl.col("Date") >= lookback_start) &
        (pl.col("Date") <= d1.date())
    )

    # chỉ cần filter Ramco Marked == "NM", bỏ compute_ramco_check
    nm_only = recent.filter(pl.col("Ramco Marked") == "NM")

    if nm_only.is_empty():
        return pd.DataFrame()

    nm_pd = nm_only.select([
        pl.col("LOB"),
        pl.col("Supervisor Name"),
        pl.col("Employee Name"),
        pl.col("Date").cast(pl.Utf8).alias("Date_str"),
    ]).to_pandas()
    nm_pd["cnt"] = 1

    pivot = nm_pd.pivot_table(
        index=["LOB","Supervisor Name","Employee Name"],
        columns="Date_str",
        values="cnt",
        aggfunc="sum",
        fill_value=0,
    ).reset_index()
    pivot.columns.name = None

    fixed_cols = ["LOB","Supervisor Name","Employee Name"]
    date_cols  = sorted([c for c in pivot.columns if c not in fixed_cols])

    pivot["Grand Total"] = pivot[date_cols].sum(axis=1)
    pivot = pivot[pivot["Grand Total"] > 0]

    for c in date_cols:
        pivot[c] = pivot[c].apply(lambda x: str(int(x)) if x > 0 else "")

    pivot["_lob_ord"] = pivot["LOB"].map(LOB_ORDER).fillna(99)
    pivot = pivot.sort_values(["_lob_ord","Supervisor Name","Employee Name"]).drop(columns=["_lob_ord"])

    return pivot[fixed_cols + date_cols + ["Grand Total"]]

# ── compute ───────────────────────────────────────────────────────────────────
atd_d1      = atd.filter(pl.col("Date") == report_date_d)
df_mismatch = compute_mismatch(atd_d1)
df_hist     = compute_historical_mismatch(atd, report_date)
df_nm       = compute_nm_pending(atd, report_date)

hist_start_s = (report_date - timedelta(days=MISMATCH_LOOKBACK_DAYS)).strftime("%d-%b-%Y")
hist_end_s   = (report_date - timedelta(days=1)).strftime("%d-%b-%Y")

print(
    f"✓ D-1: {report_date_d} | "
    f"Mismatch D-1: {len(df_mismatch)} | "
    f"Historical ({MISMATCH_LOOKBACK_DAYS}d): {len(df_hist)} | "
    f"NM Pending: {len(df_nm)}"
)

# ══════════════════════════════════════════════════════════════════════════════
# STYLE CONSTANTS
# ══════════════════════════════════════════════════════════════════════════════
HDR_DARK  = "#1a3a5c"
HDR_MID   = "#1f5c99"
HDR_MISM  = "#8b0000"
HDR_HIST  = "#7b3f00"   # nâu đậm — historical mismatch section
HDR_NM    = "#7a5200"
BLU_ROW   = "#dce8f5"
WHT_ROW   = "#ffffff"
MISS_BG   = "#fde8ea";  MISS_FG = "#9b1c2a"
WARN_BG   = "#fff3cd";  WARN_FG = "#7a5200"
TOT_BG    = "#1a3a5c"
FONT      = "font-family:Arial,sans-serif;font-size:11px;"
TH_S      = f"{FONT}padding:5px 8px;color:#fff;font-weight:bold;white-space:nowrap;text-align:center;border:1px solid rgba(255,255,255,0.2);"
TD_S      = f"{FONT}padding:4px 8px;border:1px solid #dce8f5;white-space:nowrap;text-align:left;"
TD_TOT    = f"{FONT}padding:4px 8px;border:1px solid rgba(255,255,255,0.15);white-space:nowrap;text-align:center;background:{TOT_BG};color:#fff;font-weight:bold;"

CSS = f"""
body{{margin:0;padding:16px;background:#fff;font-family:Arial,sans-serif}}
.t{{border-collapse:collapse;font-size:11px;font-family:Arial,sans-serif;
   white-space:nowrap;width:auto}}
.t thead th{{padding:5px 8px;color:#fff;font-weight:bold;
   text-align:center;border:1px solid rgba(255,255,255,0.2)}}
.t tbody td{{padding:4px 8px;border:1px solid #dce8f5;text-align:left}}
.t tbody tr.blu td{{background:{BLU_ROW}}}
.t tbody tr.wht td{{background:{WHT_ROW}}}
.t tbody tr.tot td{{background:{TOT_BG}!important;color:#fff!important;
   font-weight:bold!important;text-align:center!important}}
.c-false{{background:{MISS_BG}!important;color:{MISS_FG}!important;font-weight:bold!important}}
.c-nm{{background:{WARN_BG}!important;color:{WARN_FG}!important;font-weight:bold!important}}
"""

# ══════════════════════════════════════════════════════════════════════════════
# SECTION HEADER
# ══════════════════════════════════════════════════════════════════════════════
def _sec_hdr(title, note, hdr_color, for_email=False):
    badge_s = (f"background:{hdr_color};color:#fff;font-size:12px;"
               f"font-weight:bold;padding:5px 12px;border-radius:3px;")
    note_s  = (f"{FONT}font-size:10.5px;color:#555;background:#f8f8f8;"
               f"border-left:3px solid {hdr_color};padding:4px 10px;"
               f"margin:3px 0 8px;border-radius:0 3px 3px 0;display:block;")
    if for_email:
        return (f'<p style="margin:16px 0 3px;">'
                f'<span style="{badge_s}">{title}</span></p>'
                f'<p style="{note_s}">{note}</p>')
    return (f'<div style="margin:16px 0 3px;">'
            f'<span style="{badge_s}">{title}</span></div>'
            f'<div style="{note_s}">{note}</div>')

# ══════════════════════════════════════════════════════════════════════════════
# TABLE 1 & 2: MISMATCH  (dùng chung cho D-1 và Historical)
# ══════════════════════════════════════════════════════════════════════════════
MISMATCH_DISPLAY = [
    "Date","Supervisor Name","IEX ID","OracleID","Employee Name","LOB",
    "Original.Shift","Attendance","Ramco Marked","Ramco_Mismatch",
    "Start Time","End Time","Remark (Str)",
    "HC Schedule","Present","Planned","Unplanned","Duration","Prod Hour",
]

def build_mismatch_table(df: pd.DataFrame, hdr_color: str, empty_label: str, for_email=False) -> str:
    t_cls = "" if for_email else 'class="t" '
    cols  = [c for c in MISMATCH_DISPLAY if c in df.columns]
    h     = [f'<table {t_cls}style="border-collapse:collapse;width:auto;{FONT}"><thead><tr>']
    for c in cols:
        h.append(f'<th style="{TH_S}background:{hdr_color};">{c}</th>')
    h.append('</tr></thead><tbody>')

    if df.empty:
        h.append(f'<tr><td colspan="{len(cols)}" style="{TD_S}text-align:center;color:#888;">'
                 f'&#9989; No mismatch cases found for {empty_label}</td></tr>')
    else:
        NUM_COLS  = {"HC Schedule","Present","Planned","Unplanned","Duration","Prod Hour","IEX ID","OracleID"}
        CTR_COLS  = {"Attendance","Ramco Marked","Ramco_Mismatch","Start Time","End Time"}
        DEC1_COLS = {"Duration", "Prod Hour"}   # ← thêm dòng này

        for i, row in df.iterrows():
            even  = i % 2 == 0
            bg_r  = BLU_ROW if even else WHT_ROW
            h.append('<tr>' if for_email else f'<tr class="{"blu" if even else "wht"}">')
            for c in cols:
                val = row.get(c, "")

                # ── Format value ──────────────────────────────────────────────
                if c in DEC1_COLS and val is not None and not (isinstance(val, float) and pd.isna(val)):
                    try:    v = f"{float(val):.1f}"
                    except: v = str(val)
                else:
                    v = str(val) if val is not None and not (isinstance(val, float) and pd.isna(val)) else "&#8212;"

                align = "right" if c in NUM_COLS else ("center" if c in CTR_COLS else "left")
                extra = ""
                if c == "Ramco_Mismatch" and v == "FALSE":
                    extra = f"background:{MISS_BG};color:{MISS_FG};font-weight:bold;"
                if for_email:
                    h.append(f'<td style="{TD_S}text-align:{align};background:{bg_r};{extra}">{v}</td>')
                else:
                    cls = "c-false" if (c=="Ramco_Mismatch" and v=="FALSE") else ""
                    h.append(f'<td class="{cls}" style="{TD_S}text-align:{align};">{v}</td>')
            h.append('</tr>')
    h.append('</tbody></table>')
    return "".join(h)

# ══════════════════════════════════════════════════════════════════════════════
# TABLE 3: NM PENDING
# ══════════════════════════════════════════════════════════════════════════════
def build_nm_table(df: pd.DataFrame, for_email=False) -> str:
    t_cls = "" if for_email else 'class="t" '
    h     = [f'<table {t_cls}style="border-collapse:collapse;width:auto;{FONT}">']

    if df.empty:
        h.append(f'<thead><tr><th style="{TH_S}background:{HDR_NM};">Status</th></tr></thead>')
        h.append(f'<tbody><tr><td style="{TD_S}text-align:center;color:#888;">'
                 f'&#9989; No NM Pending cases found</td></tr></tbody></table>')
        return "".join(h)

    fixed = ["LOB","Supervisor Name","Employee Name"]
    dcols = [c for c in df.columns if c not in fixed + ["Grand Total"]]
    acols = fixed + dcols + ["Grand Total"]

    # Header
    h.append('<thead><tr>')
    for c in acols:
        if c not in df.columns: continue
        bg = HDR_NM if c=="Grand Total" else (HDR_DARK if c in fixed else HDR_MID)
        h.append(f'<th style="{TH_S}background:{bg};">{c}</th>')
    h.append('</tr></thead><tbody>')

    # Data rows
    for i, row in df.iterrows():
        even  = i % 2 == 0
        bg_r  = BLU_ROW if even else WHT_ROW
        h.append('<tr>' if for_email else f'<tr class="{"blu" if even else "wht"}">')
        for c in acols:
            if c not in df.columns: continue
            val = row.get(c,"")
            v   = str(val) if val is not None and not (isinstance(val,float) and pd.isna(val)) else ""

            if c == "Grand Total":
                if for_email:
                    h.append(f'<td style="{TD_TOT}">{v}</td>')
                else:
                    h.append(f'<td class="tot" style="{TD_S}text-align:center;font-weight:bold;">{v}</td>')
            elif c in fixed:
                if for_email:
                    h.append(f'<td style="{TD_S}background:{bg_r};">{v}</td>')
                else:
                    h.append(f'<td style="{TD_S}">{v}</td>')
            else:
                has_nm = v.strip().isdigit() and int(v) > 0
                if has_nm:
                    if for_email:
                        h.append(f'<td style="{TD_S}text-align:center;background:{WARN_BG};color:{WARN_FG};font-weight:bold;">{v}</td>')
                    else:
                        h.append(f'<td class="c-nm" style="{TD_S}text-align:center;">{v}</td>')
                else:
                    if for_email:
                        h.append(f'<td style="{TD_S}text-align:center;background:{bg_r};">{v}</td>')
                    else:
                        h.append(f'<td style="{TD_S}text-align:center;">{v}</td>')
        h.append('</tr>')

    # Grand total row
    h.append('<tr class="tot">' if not for_email else '<tr>')
    for c in acols:
        if c not in df.columns: continue
        if c in fixed:
            v = "Grand Total" if c=="LOB" else ""
            if for_email:
                h.append(f'<td style="{TD_TOT}text-align:left;">{v}</td>')
            else:
                h.append(f'<td class="tot" style="text-align:left;">{v}</td>')
        else:
            try:
                total = df[c].apply(lambda x: int(x) if str(x).strip().isdigit() else 0).sum()
                v = str(int(total)) if total > 0 else ""
            except:
                v = ""
            if for_email:
                h.append(f'<td style="{TD_TOT}">{v}</td>')
            else:
                h.append(f'<td class="tot">{v}</td>')
    h.append('</tr>')
    h.append('</tbody></table>')
    return "".join(h)

# ══════════════════════════════════════════════════════════════════════════════
# BUILD ALL SECTIONS
# ══════════════════════════════════════════════════════════════════════════════
def build_all(for_email=False):
    parts = []

    # ── Banner ────────────────────────────────────────────────────────────────
    banner_c = "#8b0000"
    b_title  = f"{FONT}font-size:14px;font-weight:bold;color:#fff;margin:0;"
    b_sub    = f"{FONT}font-size:10.5px;color:#fff;margin:3px 0 0;"
    b_inner  = (
        f'<p style="{b_title}">&#9888; Expedia VN — Attendance Mismatch &amp; NM Report</p>'
        f'<p style="{b_sub}">As of: <strong>{report_date_s}</strong> &nbsp;|&nbsp; '
        f'D-1 mismatch + Historical ({MISMATCH_LOOKBACK_DAYS}d) + NM Pending (30d)</p>'
    )
    if for_email:
        parts.append(
            f'<table width="100%" border="0" cellspacing="0" cellpadding="0" style="margin:0 0 14px;">'
            f'<tr><td style="background:{banner_c};padding:10px 14px;border-radius:4px;">'
            f'{b_inner}</td></tr></table>'
        )
    else:
        parts.append(
            f'<div style="background:{banner_c};padding:10px 14px;border-radius:4px;'
            f'margin:0 0 14px;">{b_inner}</div>'
        )

    # ── Section 1: Mismatch D-1 ───────────────────────────────────────────────
    n_mm = len(df_mismatch)
    parts.append(_sec_hdr(
        "1. Mismatch Attendance | Ramco  (D-1)",
        f"Date: {report_date_s} &nbsp;|&nbsp; "
        f"Cases where ATD Attendance ≠ Ramco Marked (Ramco_Check = <strong>FALSE</strong>). "
        f"Total: <strong style='color:{MISS_FG}'>{n_mm} case(s)</strong>.",
        HDR_MISM, for_email
    ))
    parts.append(build_mismatch_table(df_mismatch, HDR_MISM, report_date_s, for_email))

    # ── Section 2: Historical Mismatch ───────────────────────────────────────
    n_hist = len(df_hist)
    parts.append(_sec_hdr(
        f"2. Historical Mismatch  (last {MISMATCH_LOOKBACK_DAYS} days)",
        f"From {hist_start_s} to {hist_end_s} &nbsp;|&nbsp; "
        f"Unresolved FALSE cases from previous days (excluding D-1 above). "
        f"Total: <strong style='color:{MISS_FG}'>{n_hist} case(s)</strong>.",
        HDR_HIST, for_email
    ))
    parts.append(build_mismatch_table(
        df_hist, HDR_HIST, f"{hist_start_s} → {hist_end_s}", for_email
    ))

    # ── Section 3: NM Pending ─────────────────────────────────────────────────
    n_nm = len(df_nm)
    parts.append(_sec_hdr(
        "3. NM Pending  (last 15 days)",
        f"Agents with Ramco = blank / NM within last 15 days up to {report_date_s}. "
        f"Total agents: <strong>{n_nm}</strong>. "
        f"Each column = a date, value = number of NM occurrences.",
        HDR_NM, for_email
    ))
    parts.append(build_nm_table(df_nm, for_email))

    return "".join(parts)

# ══════════════════════════════════════════════════════════════════════════════
# EMAIL GREETING / SIGNATURE
# ══════════════════════════════════════════════════════════════════════════════
def build_greeting():
    return f"""
<p style="{FONT}font-size:12px;margin:0 0 10px;line-height:1.7">Dear team,</p>
<p style="{FONT}font-size:12px;margin:0 0 10px;line-height:1.7">
    Please find attached the NM Report as of <strong>{report_date_s}</strong>,
    for all active GCs.
</p>
<p style="{FONT}font-size:12px;margin:0 0 16px;line-height:1.7">
    Kindly review all case mismatches in attendance, compare with the Ramco code
    to ensure accuracy, and make sure that all pending NM cases are addressed
    at the earliest convenience.
</p>
<hr style="border:none;border-top:1px solid #e0e0e0;margin:0 0 14px;">
"""

def build_signature():
    return f"""
<hr style="border:none;border-top:1px solid #e0e0e0;margin:14px 0 10px;">
<p style="{FONT}font-size:12px;margin:0 0 4px;line-height:1.7">Thanks &amp; Regards,</p>
<p style="{FONT}font-size:12px;font-weight:bold;margin:0 0 2px;">Chinh Nguyen</p>
<p style="{FONT}font-size:11px;color:#555;margin:0 0 2px;">Analyst, WFM Real Time Management</p>
<br>
<p style="{FONT}font-size:11px;color:#555;margin:0 0 2px;line-height:1.6">
    Level 4, Tower 1, OneHub Saigon, Lot C1-2, D1 Street, Saigon Hi Tech Park,<br>
    Tan Phu Ward, District 9, Ho Chi Minh City, Vietnam
</p>
<p style="{FONT}font-size:11px;color:#555;margin:0;">
    Ph No: +84 986 473 419 &nbsp;|&nbsp;
    Email: <a href="mailto:huuchinh.nguyen@concentrix.com"
       style="color:#1f5c99;font-weight:bold;text-decoration:none;">
       huuchinh.nguyen@concentrix.com</a>
</p>
<p style="{FONT}font-size:10px;color:#aaa;margin-top:10px;">
    Generated: {datetime.now().strftime("%Y-%m-%d %H:%M")} &nbsp;|&nbsp;
    Source: ATD_Final.parquet
</p>
"""

# ══════════════════════════════════════════════════════════════════════════════
# DISPLAY NOTEBOOK
# ══════════════════════════════════════════════════════════════════════════════
if DISPLAY_NOTEBOOK:
    nb_html = (
        "<!DOCTYPE html><html><head><meta charset='utf-8'>"
        f"<style>{CSS}</style></head><body>"
        + build_all(for_email=False)
        + "</body></html>"
    )
    escaped = nb_html.replace("&","&amp;").replace('"',"&quot;").replace("'","&#39;")
    display(HTML(
        f'<iframe srcdoc="{escaped}" style="width:100%;border:none;min-height:700px;" '
        f'onload="this.style.height=(this.contentDocument.body.scrollHeight+40)+\'px\'"></iframe>'
    ))
    print("✓ Display done")

# ══════════════════════════════════════════════════════════════════════════════
# SEND EMAIL
# ══════════════════════════════════════════════════════════════════════════════
if SEND_EMAIL:
    import win32com.client

    email_html = (
        "<!--[if mso]><xml><o:OfficeDocumentSettings><o:AllowPNG/>"
        "<o:PixelsPerInch>96</o:PixelsPerInch></o:OfficeDocumentSettings></xml><![endif]-->"
        f"<div style='padding:20px 24px;background:#fff;{FONT}'>"
        + build_greeting()
        + build_all(for_email=True)
        + build_signature()
        + "</div>"
    )

    def send_auto(to, cc, subject, html_body, quit_after=True):
        pythoncom.CoInitialize()
        was_on = any(p.name().lower()=="outlook.exe" for p in psutil.process_iter(["name"]))
        if not was_on:
            for exe in [
                r"C:\Program Files\Microsoft Office\root\Office16\OUTLOOK.EXE",
                r"C:\Program Files (x86)\Microsoft Office\root\Office16\OUTLOOK.EXE",
            ]:
                if os.path.exists(exe): subprocess.Popen([exe]); break
            print("⏳ Starting Outlook...")
            for _ in range(30):
                time.sleep(1)
                try: win32com.client.GetActiveObject("Outlook.Application"); break
                except: pass
        try:
            ol = win32com.client.Dispatch("Outlook.Application")
            ol.GetNamespace("MAPI").Logon()
            mail = ol.CreateItem(0)
            mail.To=to; mail.CC=cc; mail.Subject=subject; mail.HTMLBody=html_body
            mail.Send()
            print(f"✓ Sent → {to}")
            time.sleep(3)
        finally:
            if quit_after and not was_on:
                try: ol.Quit(); print("✓ Outlook closed")
                except: pass

    send_auto(EMAIL_TO, EMAIL_CC, EMAIL_SUBJECT, email_html, quit_after=True)


📂 Loading ATD_Final.parquet...
✓ 32,180 rows
✓ AUTO MODE: using D-1 = 2026-08-05
✓ Subject: Expedia VN - Attendance Mismatch & NM Report - as of the 5th of Aug'26
✓ D-1: 2026-08-05 | Mismatch D-1: 1 | Historical (15d): 3 | NM Pending: 80


c:\Users\huuchinh.nguyen\AppData\Local\anaconda3\Lib\site-packages\IPython\core\display.py:431: UserWarning: Consider using IPython.display.IFrame instead
  warnings.warn("Consider using IPython.display.IFrame instead")


✓ Display done
✓ Sent → puneet.suneja@concentrix.com;kirpan.patar@concentrix.com;ML.HOC.Expedia.Hierarchy@concentrix.com
